# Using Callbacks in SocialJax (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/SocialJax/blob/main/tutorials/04_callbacks_colab.ipynb)

This tutorial shows you how to use callbacks for logging, checkpointing, and monitoring during training.

## Overview

Callbacks are hooks that are called at specific points during training:

- Training lifecycle: on_training_start, on_training_end
- Rollout collection: on_rollout_start, on_rollout_end
- Updates: on_update_start, on_update_end
- Steps: on_step

SocialJax provides several built-in callbacks for common tasks.

## 0. Colab Setup

In [ ]:
# @title Colab Setup { display-mode: "form" }

import os
import sys

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    print("Installing SocialJax on Google Colab...")
    
    !git clone https://github.com/cooperativex/SocialJax.git
    %cd SocialJax
    
    !pip install -q -r requirements.txt
    !pip install -q "jax[cuda11_cudnn86]" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html
    
    sys.path.insert(0, '/content/SocialJax/socialjax')
    print("\u2705 SocialJax installed successfully!")
else:
    print("Not running on Colab. Make sure SocialJax is installed locally.")
    sys.path.insert(0, './socialjax')

In [ ]:
# @title Check GPU Availability
import jax
print(f"JAX version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

## 1. Import Required Components

In [ ]:
import os
import jax

from socialjax.training import (
    Trainer,
    BaseCallback,
    CallbackList,
    CheckpointCallback,
    EvalCallback,
    ProgressCallback,
)
from socialjax import make

## 2. Built-in Callbacks Overview

In [ ]:
print("Built-in Callbacks:")
print("")
print("1. CheckpointCallback - Save model checkpoints")
print("   - save_freq: How often to save (in updates)")
print("   - save_path: Directory to save checkpoints")
print("")
print("2. EvalCallback - Periodic evaluation")
print("   - eval_env: Environment for evaluation")
print("   - eval_freq: How often to evaluate")
print("   - n_eval_episodes: Number of episodes to run")
print("")
print("3. ProgressCallback - Progress bar display")
print("   - total_timesteps: Total training timesteps")
print("   - progress_freq: How often to update progress")
print("")
print("4. WandbCallback - Weights and Biases logging")
print("   - project: Wandb project name")
print("   - name: Run name")

## 3. Using CheckpointCallback

In [ ]:
checkpoint_callback = CheckpointCallback(
    save_freq=100,
    save_path="checkpoints/ippo_coin_game",
    name_prefix="ippo",
    verbose=1,
)

print("CheckpointCallback created:")
print(f"  Save frequency: every {checkpoint_callback.save_freq} updates")
print(f"  Save path: {checkpoint_callback.save_path}")

## 4. Using EvalCallback

In [ ]:
eval_env = make('coin_game', num_agents=5)

eval_callback = EvalCallback(
    eval_env=eval_env,
    eval_freq=50,
    n_eval_episodes=10,
    best_model_save_path="checkpoints/best_model",
    deterministic=True,
    verbose=1,
)

print("EvalCallback created:")
print(f"  Eval frequency: every {eval_callback.eval_freq} updates")
print(f"  Episodes per eval: {eval_callback.n_eval_episodes}")

## 5. Using ProgressCallback

In [ ]:
progress_callback = ProgressCallback(
    total_timesteps=100000,
    progress_freq=10,
    show_metrics=['loss', 'episode_return'],
    verbose=True,
)

print("ProgressCallback created")

## 6. Combining Callbacks with CallbackList

In [ ]:
callbacks = CallbackList([
    checkpoint_callback,
    eval_callback,
    progress_callback,
])

print(f"CallbackList with {len(callbacks)} callbacks")
for i, cb in enumerate(callbacks):
    print(f"  {i+1}. {type(cb).__name__}")

## 7. Creating a Custom Callback

In [ ]:
class CustomLoggingCallback(BaseCallback):
    """Custom callback for logging training progress."""
    
    def __init__(self, log_freq=100, verbose=0):
        super().__init__(verbose)
        self.log_freq = log_freq
        self.episode_returns = []
    
    def on_training_start(self, trainer):
        """Called when training starts."""
        self.trainer = trainer
        if self.verbose:
            print("Training started!")
    
    def on_step(self, step, timestep, obs, action, reward, done, info):
        """Called after each environment step."""
        if done:
            self.episode_returns.append(info.get('episode_return', 0))
    
    def on_update_end(self, update_step, metrics):
        """Called after each policy update."""
        if update_step % self.log_freq == 0:
            if self.episode_returns:
                mean_return = sum(self.episode_returns) / len(self.episode_returns)
                print(f"Update {update_step}: Mean episode return = {mean_return:.2f}")
    
    def on_training_end(self):
        """Called when training ends."""
        if self.verbose:
            print(f"Training complete! Total episodes: {len(self.episode_returns)}")


custom_callback = CustomLoggingCallback(log_freq=50, verbose=1)
print("Custom callback created!")

## 8. Using Callbacks with Trainer

In [ ]:
print("Using callbacks with Trainer:")
print("")
print("# Method 1: Pass callbacks to Trainer")
print("trainer = Trainer(")
print("    algorithm='ippo',")
print("    env='coin_game',")
print("    callbacks=[checkpoint_callback, eval_callback],")
print(")")
print("")
print("# Method 2: Pass callbacks to train()")
print("metrics = trainer.train(")
print("    total_timesteps=100000,")
print("    callbacks=callbacks,")
print(")")

## 9. Callback Hook Reference

In [ ]:
print("Callback Hook Reference:")
print("")
print("on_training_start(trainer)")
print("  - Called at the beginning of training")
print("  - Parameters: trainer instance")
print("")
print("on_training_end()")
print("  - Called at the end of training")
print("")
print("on_rollout_start()")
print("  - Called before collecting each rollout")
print("")
print("on_rollout_end(rollout_data)")
print("  - Called after collecting each rollout")
print("  - Parameters: rollout data dict")
print("")
print("on_step(step, timestep, obs, action, reward, done, info)")
print("  - Called after each environment step")
print("")
print("on_update_start(update_step)")
print("  - Called before each policy update")
print("")
print("on_update_end(update_step, metrics)")
print("  - Called after each policy update")
print("  - Parameters: update step number, metrics dict")

## 10. Complete Training Example

In [ ]:
print("Complete training example with callbacks:")
print("")
print("""from socialjax.training import Trainer, CheckpointCallback, EvalCallback, ProgressCallback
from socialjax import make

# Create callbacks
checkpoint_cb = CheckpointCallback(
    save_freq=500,
    save_path="checkpoints/experiment",
    verbose=1,
)

eval_env = make('coin_game', num_agents=5)
eval_cb = EvalCallback(
    eval_env=eval_env,
    eval_freq=200,
    n_eval_episodes=20,
    deterministic=True,
)

progress_cb = ProgressCallback(
    total_timesteps=1_000_000,
)

# Create trainer with callbacks
trainer = Trainer(
    algorithm='ippo',
    env='coin_game',
    num_agents=5,
)

# Train with callbacks
metrics = trainer.train(
    total_timesteps=1_000_000,
    callbacks=[checkpoint_cb, eval_cb, progress_cb],
)
""")

## Summary

In [ ]:
print("Summary:")
print("")
print("In this tutorial, you learned:")
print("")
print("1. Built-in callbacks: CheckpointCallback, EvalCallback, ProgressCallback, WandbCallback")
print("2. How to create custom callbacks by inheriting from BaseCallback")
print("3. Callback hook points and their parameters")
print("4. How to combine multiple callbacks using CallbackList")
print("5. How to use callbacks with the Trainer class")